<a href="https://colab.research.google.com/github/frank-morales2020/Cloud_curious/blob/master/FineTuning_LLM_Mistral_7B_Instruct_v0_1_for_text_to_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

v0.1:    https://medium.com/thedeephub/fine-tuning-the-llm-mistral-7b-for-text-to-sql-with-sql-create-context-dataset-4e9234f7691c?postPublishedType=repub

As of April 2026, the latest and most advanced version of the 7B instruct family is **Mistral-7B-Instruct-v0.3**.

While the original `v0.1` was a breakthrough for the 7B-parameter class, Mistral AI has released two major updates since then that significantly improved its capabilities.

### Major Versions Compared

| Feature | v0.1 (Original) | v0.2 | v0.3 (Latest) |
| :--- | :--- | :--- | :--- |
| **Context Window** | 8K tokens | 32K tokens | **32K tokens** |
| **Tokenizer** | v1 | v2 | **v3 (extended)** |
| **Vocabulary Size** | 32,000 | 32,000 | **32,768** |
| **Function Calling** | No | No | **Yes (Native)** |
| **Attention Mechanism** | Sliding Window | No SWA | **No SWA** |

---

### Why move to v0.3?

1.  **Native Function Calling:** Unlike `v0.1`, which required complex prompting or wrappers to interact with external tools, `v0.3` supports function calling natively. This is a game-changer if you're building agents or RAG systems.
2.  **Extended Vocabulary:** The v3 tokenizer increases the vocabulary size to 32,768, which makes it more efficient at handling different languages and specialized technical terms.
3.  **Removal of Sliding Window Attention (SWA):** While SWA was an interesting optimization in `v0.1`, `v0.3` follows the `v0.2` trend of removing it to improve performance consistency and simplify integration with various inference engines (like `vLLM` or `llama.cpp`).

### How to get it
You can find the official weights on Hugging Face under the identifier: `mistralai/Mistral-7B-Instruct-v0.3`.

If you're using **Ollama**, you can pull the latest version by simply running:
`ollama run mistral` (which usually defaults to the latest 7B tag).

Are you planning to deploy this locally on one of your edge devices, or are you looking to fine-tune it for a specific task?

In [ ]:
# Install Pytorch & other libraries
!pip install torch tensorboard --quiet

# Install Hugging Face libraries
!pip install  --upgrade transformers datasets accelerate evaluate bitsandbytes --quiet

#FlashAttention only supports Ampere GPUs or newer. #NEED A100 IN GOOGLE COLAB
#!pip install -U transformers
#!pip install -U flash-attn --no-build-isolation --quiet

!pip install -q https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

! pip install peft --quiet
! pip install datasets trl ninja packaging --quiet

# Uncomment only if you're using A100 GPU
#!pip install flash-attn --no-build-isolation
!pip install diffusers safetensors  --quiet
!pip install colab-env --quiet


In [ ]:
!pip install --upgrade trl transformers accelerate peft bitsandbytes -q

In [21]:
import colab_env
import os

access_token_write = os.environ.get("HUGGINGFACE_ACCESS_TOKEN_WRITE")

In [ ]:
print(access_token_write)

In [23]:
!ls

gdrive	sample_data


In [25]:
from huggingface_hub import login

import os
os.environ.pop("HF_TOKEN", None)

login(
  token=access_token_write,
  add_to_git_credential=True
)

In [26]:
import torch
import os
import sys
import json
import IPython
from datetime import datetime
from datasets import load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoTokenizer,
    TrainingArguments,
)
from trl import SFTTrainer

In [28]:
# set device
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
!apt-get update && apt-get install -y cuda-11-0

In [32]:
!python --version
!nvcc --version
!nvidia-smi

Python 3.12.13
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Tue Apr 14 18:27:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   3

In [33]:
import torch
torch.__version__

'2.10.0+cu128'

In [1]:
import torch
import os
import sys
import json
import IPython
from datetime import datetime
from datasets import load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoTokenizer,
    TrainingArguments,
)
from trl import SFTTrainer

In [ ]:
from datasets import load_dataset

# Convert dataset to OAI messages
system_message = """You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.
SCHEMA:
{schema}"""

def create_conversation(sample):
  return {
    "messages": [
      {"role": "system", "content": system_message.format(schema=sample["context"])},
      {"role": "user", "content": sample["question"]},
      {"role": "assistant", "content": sample["answer"]}
    ]
  }

# Load dataset from the hub
dataset = load_dataset("b-mc2/sql-create-context", split="train")
dataset = dataset.shuffle().select(range(12500))

# Convert dataset to OAI messages
dataset = dataset.map(create_conversation, remove_columns=dataset.features,batched=False)

# split dataset into 10,000 training samples and 2,500 test samples
dataset = dataset.train_test_split(test_size=2500/12500)

print(dataset["train"][345]["messages"])

# save datasets to disk
dataset["train"].to_json("train_dataset.json", orient="records")
dataset["test"].to_json("test_dataset.json", orient="records")

In [ ]:
from datasets import load_dataset

# Load jsonl data from disk for sql
dataset = load_dataset("json", data_files="train_dataset.json", split="train")
eval_dataset = load_dataset("json", data_files="test_dataset.json", split="train")

In [4]:
print(dataset[345]["messages"])

[{'content': 'You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.\nSCHEMA:\nCREATE TABLE table_name_72 (venue VARCHAR, score VARCHAR)', 'role': 'system'}, {'content': 'Where was the game played when the score was 1-4?', 'role': 'user'}, {'content': 'SELECT venue FROM table_name_72 WHERE score = "1-4"', 'role': 'assistant'}]


In [5]:
print(eval_dataset[345]["messages"])

[{'content': 'You are an text to SQL query translator. Users will ask you questions in English and you will generate a SQL query based on the provided SCHEMA.\nSCHEMA:\nCREATE TABLE table_name_14 (location VARCHAR, home_ground VARCHAR)', 'role': 'system'}, {'content': 'Which Location has a Home Ground of Mong kok stadium?', 'role': 'user'}, {'content': 'SELECT location FROM table_name_14 WHERE home_ground = "mong kok stadium"', 'role': 'assistant'}]


https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import logging

# Set logging to error only for transformers and peft
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("peft").setLevel(logging.ERROR)



# 1. Device Setup
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Model ID (Mistral v0.3 is the current stable version)
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# 3. BitsAndBytes Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 4. Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
    trust_remote_code=True
)

# 5. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

# 6. Manual ChatML Setup (Replacing setup_chat_format)
# Define the ChatML tokens
chatml_tokens = {"additional_special_tokens": ["<|im_start|>", "<|im_end|>"]}
tokenizer.add_special_tokens(chatml_tokens)

# Set the pad token (using unk_token or im_end is common for Mistral)
tokenizer.pad_token = "<|im_end|>"
tokenizer.padding_side = 'right'

# 7. Resize Embeddings
# Crucial: This accommodates the new <|im_start|> and <|im_end|> tokens
model.resize_token_embeddings(len(tokenizer))

# 8. Define the Jinja Template for ChatML
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% endif %}"
)

print(f"Successfully loaded {model_id} on {device}")
print(f"Tokenizer size updated to: {len(tokenizer)}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Successfully loaded mistralai/Mistral-7B-Instruct-v0.3 on cuda
Tokenizer size updated to: 32770


In [7]:
print(model)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32770, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (n

In [8]:
from peft import LoraConfig

# LoRA config based on QLoRA paper & Sebastian Raschka experiment
peft_config = LoraConfig(
        lora_alpha=128,
        lora_dropout=0.05,
        r=256,
        bias="none",
        target_modules="all-linear",
        task_type="CAUSAL_LM",
)

Administration routines for huggingface.co

In [ ]:
#!pip install huggingface_hub --quiet
from huggingface_hub import HfApi

api = HfApi()
api.get_token_permission(token=access_token_write)
#api.set_access_token(access_token)
# frankmorales2020/Mistral-7B-text-to-sql Good


repo_id = "my-awesome-model-poc"

#frankmorales2020/Mistral-7B-squad2
#api.create_repo(repo_id=repo_id, private=False)

#api.delete_repo(repo_id=repo_id)


#api.upload_folder(
#    folder_path="./model",
#    repo_id=repo_id,
#    repo_type="model",
#)


In [ ]:
from trl import SFTTrainer, SFTConfig

import logging

# Set logging to error only for transformers and peft
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("peft").setLevel(logging.ERROR)


# 1. Define SFTConfig
sft_config = SFTConfig(
    output_dir="Mistral-7B-v0dot3-text-to-sql-flash-attention-2",
    push_to_hub=True,
    report_to="tensorboard",

    # --- Evaluation Logic (The Fix) ---
    eval_strategy="steps",          # REPLACES evaluation_strategy
    eval_steps=25,                  # How often to compute eval loss
    do_eval=True,

    # --- SFT Specifics ---
    max_length=3072,
    packing=True,
    dataset_text_field="text",

    # --- Training Hyperparameters ---
    num_train_epochs=3,
    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,   # Added for evaluation
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    warmup_steps=100,
    max_grad_norm=0.3,

    # --- Precision ---
    bf16=True,
    tf32=True,
    logging_steps=25,
    save_strategy="epoch"
)

# 2. Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=sft_config,
)

# 3. Final Token Alignment (Based on your update)
# This ensures the model's generation config matches your new pad_token_id
model.config.pad_token_id = 32769
model.generation_config.pad_token_id = 32769

In [ ]:
# start training, the model will be automatically saved to the hub and the output directory
trainer.train()

# save model
trainer.save_model()

{'loss': '1.213', 'grad_norm': '0.2275', 'learning_rate': '0.0002', 'entropy': '1.181', 'num_tokens': '4.426e+05', 'mean_token_accuracy': '0.7978', 'epoch': '0.365'}


In [ ]:
# After trainer.train() finishes
trainer.save_model("Mistral-7B-v0dot3-text-to-sql-final")
tokenizer.save_pretrained("Mistral-7B-v0dot3-text-to-sql-final")

In [ ]:
# free the memory again
del model
del trainer
torch.cuda.empty_cache()

## Test Model and run Inference

In [ ]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline
%cd /content/
peft_model_id = "./Mistral-7B-v0dot3-text-to-sql-flash-attention-2"

# Load Model with PEFT adapter
model = AutoPeftModelForCausalLM.from_pretrained(
  peft_model_id,
  device_map="auto",
  torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
# load into pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Let’s load our test dataset try to generate an instruction.

In [ ]:
print(model)

In [ ]:
from datasets import load_dataset
from random import randint


# Load our test dataset
eval_dataset = load_dataset("json", data_files="test_dataset.json", split="train")
rand_idx = randint(0, len(eval_dataset))

# Test on sample
prompt = pipe.tokenizer.apply_chat_template(eval_dataset[rand_idx]["messages"][:2], tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, do_sample=False, temperature=0.1, top_k=50, top_p=0.1, eos_token_id=pipe.tokenizer.eos_token_id, pad_token_id=pipe.tokenizer.pad_token_id)

print(f"Query:\n{eval_dataset[rand_idx]['messages'][1]['content']}")
print(f"Original Answer:\n{eval_dataset[rand_idx]['messages'][2]['content']}")
print(f"Generated Answer:\n{outputs[0]['generated_text'][len(prompt):].strip()}")

In [ ]:
from tqdm import tqdm

def evaluate(sample):
    prompt = pipe.tokenizer.apply_chat_template(sample["messages"][:2], tokenize=False, add_generation_prompt=True)
    outputs = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7, top_k=50, top_p=0.95, eos_token_id=pipe.tokenizer.eos_token_id, pad_token_id=pipe.tokenizer.pad_token_id)
    predicted_answer = outputs[0]['generated_text'][len(prompt):].strip()
    if predicted_answer == sample["messages"][2]["content"]:
        return 1
    else:
        return 0

success_rate = []
number_of_eval_samples = 1000
# iterate over eval dataset and predict
for s in tqdm(eval_dataset.shuffle().select(range(number_of_eval_samples))):
    success_rate.append(evaluate(s))

# compute accuracy
accuracy = sum(success_rate)/len(success_rate)

print(f"Accuracy: {accuracy*100:.2f}%")

  1%|          | 9/1000 [00:18<38:31,  2.33s/it]/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1157: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
100%|██████████| 1000/1000 [35:30<00:00,  2.13s/it]

Accuracy: 79.50%


When evaluated on 1000 samples from the evaluation dataset, our model achieved an impressive accuracy of 79.50%. However, there's room for improvement. We could enhance the model's performance by exploring techniques like few-shot learning, RAG, and Self-healing to generate the SQL query.



## Deploy the LLM for Production WITH dockers

In [ ]:
import locale
locale.setlocale(locale.LC_ALL, 'en_US.UTF-8')

'en_US.UTF-8'

In [ ]:
%%shell
pip install udocker
udocker --allow-root install

In [ ]:
%%shell

sudo apt update -qq
sudo apt install apt-transport-https ca-certificates curl software-properties-common -qq
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo apt-key add -
sudo add-apt-repository "deb [arch=amd64] https://download.docker.com/linux/ubuntu bionic stable"
sudo apt update -qq
sudo apt install docker-ce
docker

In [ ]:
%%shell

docker --version

In [ ]:
%%shell
set -x
dockerd -b none --iptables=0 -l warn &
for i in $(seq 5); do [ ! -S "/var/run/docker.sock" ] && sleep 2 || break; done
docker info
docker network ls
docker pull hello-world
docker pull ubuntu
# docker build -t myimage .
docker images
kill $(jobs -p)

In [ ]:
%%shell

docker --version

model=/content/Mistral-7B-text-to-sql
num_shard=1             # number of shards
max_input_length=1024   # max input length
max_total_tokens=2048   # max total tokens

docker run -d --name tgi --gpus all -ti -p 8080:80 \
  -e MODEL_ID=/workspace \
  -e NUM_SHARD=$num_shard \
  -e MAX_INPUT_LENGTH=max_input_length \
  -e MAX_TOTAL_TOKENS=max_total_tokens \
  -v $model:/workspace \
  ghcr.io/huggingface/text-generation-inference:latest


Once your container is running you can send requests.

In [ ]:
import requests as r
from transformers import AutoTokenizer
from datasets import load_dataset
from random import randint

# Load our test dataset and Tokenizer again
#tokenizer = AutoTokenizer.from_pretrained("Mistral-7B-text-to-sql")
tokenizer = AutoTokenizer.from_pretrained("Mistral-7B-text-to-sql-flash-attention-2")
#tokenizer = AutoTokenizer.from_pretrained("Smaug-72B-v0.1")
eval_dataset = load_dataset("json", data_files="test_dataset.json", split="train")
rand_idx = randint(0, len(eval_dataset))

# generate the same prompt as for the first local test
prompt = tokenizer.apply_chat_template(eval_dataset[rand_idx]["messages"][:2], tokenize=False, add_generation_prompt=True)
request= {"inputs":prompt,"parameters":{"temperature":0.2, "top_p": 0.95, "max_new_tokens": 256}}

# send request to inference server
resp = r.post("http://127.0.0.1:8080/generate", json=request)

print(resp)
output = resp.json()["generated_text"].strip()
time_per_token = resp.headers.get("x-time-per-token")
time_prompt_tokens = resp.headers.get("x-prompt-tokens")

# Print results
print(f"Query:\n{eval_dataset[rand_idx]['messages'][1]['content']}")
print(f"Original Answer:\n{eval_dataset[rand_idx]['messages'][2]['content']}")
print(f"Generated Answer:\n{output}")
print(f"Latency per token: {time_per_token}ms")
print(f"Latency prompt encoding: {time_prompt_tokens}ms")

Awesome, Don't forget to stop your container once you are done.

In [ ]:
!docker stop tgi

Deploy the LLM for Production WITH huggingface.co
https://github.com/frank-morales2020/MLxDL/blob/main/upload_model_hf.ipynb